# Módulo 2 — Aplicação do NFStream em um PCAP do CICIDS2017
**Objetivo:** Aplicar o NFStream sobre um arquivo PCAP original do CICIDS2017, gerar um CSV de fluxos
e comparar o resultado com o CSV oficial disponibilizado pelos autores do dataset.

**Dia/arquivo escolhido:** `Thursday-WorkingHours-Morning-WebAttacks`
(mesmo dia usado no notebook de EDA da aula 2, o que facilita a comparação).

**Autor:** Marcel Teixeira Chiarelo
**Data:** julho/2026


In [2]:
from nfstream import NFStreamer
import pandas as pd

# Caminho do PCAP baixado (ajuste conforme sua máquina)
PCAP_PATH = "dados/pcaps/Thursday-WorkingHours.pcap"

# Caminho do CSV original do CICIDS2017 para o mesmo dia/janela, já usado no modulo2/dados
CSV_ORIGINAL_PATH = "dados/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"

CSV_SAIDA_NFSTREAM = "captura_nfstream_thursday_webattacks.csv" 

## 1. Aplicar o NFStream sobre o PCAP

Diferente do Módulo 2 anterior (captura ao vivo da interface de rede), aqui o NFStream lê um
arquivo `.pcap` já existente como fonte (`source=PCAP_PATH`), processando os pacotes offline
e agregando-os em fluxos bidirecionais.

In [ ]:
print("Processando PCAP com NFStream... isso pode levar alguns minutos dependendo do tamanho do arquivo.")

streamer = NFStreamer(
    source=PCAP_PATH,
    statistical_analysis=True,
    idle_timeout=15,     # mesmo padrão usado no CICFlowMeter (timeout de fluxo em segundos)
    active_timeout=120
)

df_nf = streamer.to_pandas()
print(f"Total de fluxos gerados pelo NFStream: {len(df_nf)}")
df_nf.head()

In [ ]:
df_nf.to_csv(CSV_SAIDA_NFSTREAM, index=False)
print(f"Arquivo salvo: {CSV_SAIDA_NFSTREAM}")
print(f"Dimensões: {df_nf.shape}")

## 2. Carregar o CSV original do CICIDS2017

Este é o CSV oficial, gerado pelos autores com o CICFlowMeter, já usado no notebook de EDA
(`EDA_aula2_referencia.ipynb`).

In [4]:
df_cic = pd.read_csv(CSV_ORIGINAL_PATH)
df_cic.columns = df_cic.columns.str.strip()  # o CSV original tem espaços à frente em vários nomes de coluna
print(f"Dimensões do CSV original CICIDS2017: {df_cic.shape}")
df_cic.head()

Dimensões do CSV original CICIDS2017: (170366, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,389,113095465,48,24,9668,10012,403,0,201.416667,203.548293,...,32,203985.500,5.758373e+05,1629110,379,13800000.0,4.277541e+06,16500000,6737603,BENIGN
1,389,113473706,68,40,11364,12718,403,0,167.117647,171.919413,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4.229413e+06,16500000,6945512,BENIGN
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.333,1.170000e+07,20400000,6,24400000.0,2.430000e+07,60100000,5702188,BENIGN
3,443,60261928,9,7,2330,4221,1093,0,258.888889,409.702161,...,20,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN
4,53,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN


In [7]:
df_cic.groupby(['Label'])['Label'].count()

Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � Sql Injection        21
Web Attack � XSS                 652
Name: Label, dtype: int64

## 3. Comparação entre os dois conjuntos de fluxos

Comparamos quatro aspectos, conforme pedido na proposta: número de fluxos, nomes das features,
tipos de dados e rótulos disponíveis.

**Resultado obtido nesta execução (PCAP `Thursday-WorkingHours.pcap`, 167 MB, vs. CSV
`Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv`):**

| Métrica | NFStream (PCAP) | CICIDS2017 original (CSV) |
|---|---|---|
| Número de fluxos | 380.135 | 170.366 |
| Número de colunas | 86 | 79 (inclui `Label`) |

A diferença de mais de 200 mil fluxos é esperada e **não indica erro**: o PCAP baixado
(`Thursday-WorkingHours.pcap`) contém o **dia inteiro** de tráfego de quinta-feira, enquanto o CSV
de referência cobre **apenas o período da manhã** (há um segundo CSV,
`Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv`, para a tarde do mesmo dia). Ou seja,
o `df_nf` soma fluxos de manhã + tarde, o que já explica a maior parte da diferença.

In [ ]:
print("=== Número de fluxos ===")
print(f"NFStream (dia inteiro): {len(df_nf)} fluxos")
print(f"CICIDS2017 original (apenas manhã): {len(df_cic)} fluxos")
print(f"Diferença: {len(df_nf) - len(df_cic)} fluxos")

In [ ]:
print("=== Número de colunas (features) ===")
print(f"NFStream : {df_nf.shape[1]} colunas")
print(f"CICIDS2017 original: {df_cic.shape[1]} colunas (inclui a coluna 'Label')")

In [ ]:
# Presença da coluna de rótulo
print("Coluna 'Label' no CSV do NFStream?", "Label" in df_nf.columns)
print("Coluna 'Label' no CSV original?", "Label" in df_cic.columns)
if "Label" in df_cic.columns:
    print()
    print(df_cic["Label"].value_counts())

### 3.1 Tentativa de alinhar por horário (manhã vs. dia inteiro)

Para isolar apenas os fluxos da manhã no `df_nf` e comparar de forma mais justa com o `df_cic`,
seria necessário cruzar pelo timestamp de início de cada fluxo (`bidirectional_first_seen_ms` no
NFStream). Ao inspecionar as colunas do CSV original, no entanto, foi encontrada uma limitação
importante:

In [ ]:
print(list(df_cic.columns))

In [ ]:
[c for c in df_cic.columns if "time" in c.lower()]

**Achado relevante:** a versão do CICIDS2017 distribuída em `MachineLearningCSV.zip` (a mesma
usada no notebook de EDA da aula 2) **não contém as colunas `Timestamp`, `Source IP`,
`Destination IP`, `Source Port` nem `Destination Port`** — restam apenas as 78 features numéricas
mais a coluna `Label`. Essas colunas de metadados foram removidas pelos autores nessa versão
"pronta para ML", provavelmente por questões de anonimização e para evitar vazamento de
identificadores triviais (IP/porta) no treinamento de modelos.

Isso torna **impossível**, usando apenas esse CSV, alinhar os fluxos do NFStream pela janela de
horário da manhã ou cruzar fluxo a fluxo com o PCAP por IP/porta/timestamp. Para isso seria
necessário baixar, em vez do `MachineLearningCSV.zip`, o pacote `GeneratedLabelledFlows.zip`
disponibilizado na mesma página do CICIDS2017, que mantém `Flow ID`, IPs, portas e timestamp
junto com o rótulo — mantendo a versão `MachineLearningCSV.zip` reservada apenas para a etapa de
modelagem (Módulos 3-6), sem os identificadores de rede.

In [ ]:
# Comparação de tipos de dados (dtypes) das colunas em comum,
# usando um mapeamento manual de nomes equivalentes entre NFStream e CICFlowMeter/CICIDS2017
mapeamento_colunas = {
    "src_ip": "Source IP",
    "dst_ip": "Destination IP",
    "src_port": "Source Port",
    "dst_port": "Destination Port",
    "protocol": "Protocol",
    "bidirectional_duration_ms": "Flow Duration",
    "bidirectional_packets": "Total Fwd Packets",   # aproximado, ver texto de discussão abaixo
    "bidirectional_bytes": "Total Length of Fwd Packets",  # aproximado
    "bidirectional_mean_ps": "Average Packet Size",
}

comparacao = []
for col_nf, col_cic in mapeamento_colunas.items():
    comparacao.append({
        "coluna_nfstream": col_nf,
        "dtype_nfstream": str(df_nf[col_nf].dtype) if col_nf in df_nf.columns else "ausente",
        "coluna_cicids": col_cic,
        "dtype_cicids": str(df_cic[col_cic].dtype) if col_cic in df_cic.columns else "ausente",
    })

pd.DataFrame(comparacao)

## 4. Discussão dos resultados

**Número de fluxos:** nesta execução o NFStream gerou 380.135 fluxos a partir do PCAP do dia
inteiro de quinta-feira, contra 170.366 fluxos no CSV original (apenas manhã) — uma diferença de
209.769 fluxos. A maior parte dessa diferença vem da granularidade dos arquivos (dia inteiro vs.
janela da manhã, discutido no item 5 abaixo), e não necessariamente de uma discrepância na forma de
extração de fluxos em si. Uma comparação numérica limpa (mesma janela de tempo dos dois lados)
exigiria a versão `GeneratedLabelledFlows.zip` do dataset, que preserva timestamp/IP/porta — a
versão `MachineLearningCSV.zip` usada aqui não permite esse recorte (ver item 3.1). Ainda assim,
mesmo com o recorte de horário, é esperado que sobre uma diferença residual, pois os timeouts de
fluxo (`idle_timeout`/`active_timeout`) usados pelo CICFlowMeter (ferramenta original dos autores)
não são exatamente os mesmos parâmetros do NFStream, mesmo tentando aproximar os valores (15s/120s).
Pequenas diferenças na definição de início/fim de fluxo (TCP FIN/RST vs. timeout) também alteram a
contagem.

**Nomes das features:** as 86 colunas do NFStream cobrem, de forma aproximada, as mesmas informações
que as ~79 colunas do CSV do CICIDS2017, mas com nomenclatura totalmente diferente e sem
correspondência 1-para-1 direta. Por exemplo, o CICIDS2017 divide os pacotes em "Fwd" e "Bwd" e
calcula estatísticas (min/mean/std/max) separadamente para cada direção, enquanto o NFStream usa o
prefixo `src2dst_`/`dst2src_`/`bidirectional_`. Algumas métricas (ex.: *Flow Bytes/s*, *Down/Up Ratio*)
não têm equivalente direto e precisariam ser recalculadas manualmente a partir das colunas brutas do
NFStream.

**Tipos de dados:** de forma geral os tipos numéricos batem (inteiros e floats), mas o CSV original
tem espaços em branco no início dos nomes de coluna (uma particularidade da versão distribuída pelos
autores) e algumas colunas de IP/porta que no NFStream vêm tipadas como string/int já normalizados.

**Rótulos:** a diferença mais importante. O CSV original já vem com a coluna `Label` preenchida
(BENIGN ou o tipo de ataque), pois os autores sabiam exatamente os horários em que cada ataque foi
executado durante a captura controlada. O NFStream, aplicado diretamente sobre o PCAP, **não gera
nenhum rótulo** — é preciso reconstruí-lo manualmente, cruzando IP de origem/destino, porta e horário
do fluxo com a documentação de horários de ataque do CICIDS2017 (ver seção 5).

## 5. Reconstrução (parcial) dos rótulos

A documentação do CICIDS2017 descreve os horários e IPs envolvidos em cada ataque (ex.: para a
quinta-feira de manhã, os ataques Web Attack – Brute Force, Web Attack – XSS e Web Attack – SQL
Injection têm origem no IP `172.16.0.1` contra o servidor `192.168.10.50`, em janelas de horário
específicas). Uma rotulagem aproximada pode ser feita assim:

In [ ]:
def rotular_fluxo(row):
    # Regra simplificada de exemplo — ajustar IPs/portas/horários exatos conforme a
    # documentação oficial do CICIDS2017 para o dia/ataque em questão.
    ATACANTE = "172.16.0.1"
    ALVO = "192.168.10.50"
    if row["src_ip"] == ATACANTE and row["dst_ip"] == ALVO:
        return "SUSPEITO_WEB_ATTACK"
    return "BENIGN"

df_nf["Label_reconstruido"] = df_nf.apply(rotular_fluxo, axis=1)
df_nf["Label_reconstruido"].value_counts()

> Esta rotulagem é apenas ilustrativa. Uma rotulagem confiável exige cruzar, para cada
> subconjunto de ataque, o IP de origem/destino, a porta e a janela de tempo exatos publicados na
> documentação do CICIDS2017 — processo manual, sujeito a erro e que não garante 100% de
> equivalência com os rótulos originais.

## 6. Dificuldades encontradas ao tentar replicar o CICIDS2017 a partir dos PCAPs

Foram encontradas as seguintes dificuldades principais ao tentar reconstruir o CSV do CICIDS2017
usando NFStream a partir do PCAP original:

1. **Granularidade dos arquivos.** O CICIDS2017 disponibiliza um único PCAP por dia (ex.: o
   `Thursday-WorkingHours.pcap` usado aqui, 167 MB, cobrindo o dia inteiro), enquanto os CSVs de
   referência são divididos por período/tipo de ataque (`Thursday-WorkingHours-Morning-WebAttacks` e
   `Thursday-WorkingHours-Afternoon-Infilteration`). Isso por si só já explica boa parte da diferença
   observada entre os 380.135 fluxos extraídos do PCAP (dia inteiro) e os 170.366 fluxos do CSV
   (apenas manhã).
2. **Versão do CSV sem IP/porta/timestamp.** Ao tentar filtrar o `df_nf` pela janela de horário da
   manhã para comparar de forma mais justa com o `df_cic`, descobrimos que a versão
   `MachineLearningCSV.zip` do CICIDS2017 (usada neste trabalho para as etapas de modelagem) **não
   contém as colunas `Timestamp`, `Source IP`, `Destination IP`, `Source Port` nem
   `Destination Port`** — só restam as 78 features numéricas e o `Label`. Essa é uma dificuldade
   real e não documentada de forma explícita nos materiais da disciplina: sem essas colunas, não é
   possível alinhar por horário nem cruzar fluxo a fluxo entre o CSV e o PCAP/NFStream usando apenas
   essa versão. Seria necessário baixar em paralelo o pacote `GeneratedLabelledFlows.zip`, que
   preserva esses metadados.
3. **Tamanho dos arquivos.** Mesmo um único dia de PCAP (167 MB neste caso) já é bem maior que o CSV
   correspondente (poucos MB), e dias com mais tráfego chegam à casa dos gigabytes — exigindo mais
   tempo de processamento e espaço em disco.
4. **Ausência de rótulos.** O NFStream não rotula os fluxos; a rotulagem exige cruzamento manual com
   a documentação de horários/IPs de ataque publicada pelos autores, um processo trabalhoso e
   propenso a pequenas divergências.
5. **Definição de fluxo diferente entre ferramentas.** O CICFlowMeter (usado pelos autores) e o
   NFStream implementam a extração de fluxos de forma distinta (parâmetros de timeout, tratamento de
   flags TCP, agregação bidirecional), então mesmo usando o mesmo PCAP não se obtém exatamente o
   mesmo número de fluxos nem os mesmos valores estatísticos.
6. **Nomenclatura e cálculo de features não são idênticos.** As 86 colunas do NFStream cobrem
   estatísticas semelhantes às 79 colunas do CICIDS2017, mas com nomes e, em alguns casos, fórmulas
   de cálculo diferentes, exigindo um trabalho de mapeamento manual (feito de forma parcial na seção 3
   acima) para tornar os dois formatos comparáveis.

Essas dificuldades explicam por que a réplica obtida aqui é **parcial**: ela reproduz a estrutura
geral de um dataset de fluxos de rede a partir de um PCAP (380.135 fluxos, 86 features), mas não é
numericamente idêntica ao CSV oficial do CICIDS2017 (170.366 fluxos, 79 features na janela da
manhã), tanto pela diferença de janela de tempo quanto pela ausência de metadados de rede na versão
do CSV usada neste trabalho.